In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
train_data_dir = 'imagedataset/dataset'

In [3]:

train_datagen = ImageDataGenerator(
    rescale=1./255.,
    rotation_range=45,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)


train_generator = train_datagen.flow_from_directory(
    train_data_dir, 
    batch_size=32,
    class_mode='binary', 
    target_size=(224, 224)
)


Found 2829 images belonging to 2 classes.


In [4]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available:  1


In [5]:
base_model = InceptionV3(input_shape=(224, 224, 3), include_top=False, weights='imagenet')

for layer in base_model.layers[:100]:
    layer.trainable = False
for layer in base_model.layers[100:]:
    layer.trainable = True

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(1024, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(512, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
x = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.models.Model(base_model.input, x)

early_stopping = EarlyStopping(monitor='acc', patience=5, restore_best_weights=True)

model.compile(optimizer=tf.keras.optimizers.RMSprop(lr=0.0001), 
              loss='binary_crossentropy', 
              metrics=['acc'])
inception_hist = model.fit(train_generator, 
                           steps_per_epoch=len(train_generator),
                           epochs=15,
                           callbacks=[early_stopping])

print("Final Accuracy: ", inception_hist.history['acc'][-1])

c:\conda\envs\tf\lib\site-packages\keras\optimizers\optimizer_v2\rmsprop.py:140: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


Epoch 1/15
89/89 [==============================] - 111s 843ms/step - loss: 0.4929 - acc: 0.7872
Epoch 2/15
89/89 [==============================] - 96s 1s/step - loss: 0.3703 - acc: 0.8392
Epoch 3/15
89/89 [==============================] - 91s 1s/step - loss: 0.2647 - acc: 0.8950
Epoch 4/15
89/89 [==============================] - 95s 1s/step - loss: 0.2033 - acc: 0.9191
Epoch 5/15
89/89 [==============================] - 92s 1s/step - loss: 0.1726 - acc: 0.9335
Epoch 6/15
89/89 [==============================] - 71s 789ms/step - loss: 0.1486 - acc: 0.9445
Epoch 7/15
89/89 [==============================] - 73s 814ms/step - loss: 0.1189 - acc: 0.9590
Epoch 8/15
89/89 [==============================] - 76s 856ms/step - loss: 0.1064 - acc: 0.9601
Epoch 9/15
89/89 [==============================] - 78s 879ms/step - loss: 0.0955 - acc: 0.9661
Epoch 10/15
89/89 [==============================] - 76s 859ms/step - loss: 0.0907 - acc: 0.9717
Epoch 11/15
89/89 [==============================]

In [6]:
model.save('inception_model.h5')